# Bernoulli Naive Bayes (Multivariate Bernoulli Event Model)

## Learning Objectives
- Understand the **multivariate Bernoulli event model** and how it differs from Gaussian NB
- Derive the MAP decision rule for binary feature vectors
- Compute **MLE estimates** for $\phi_{kj}$ with and without Laplace smoothing
- Implement a vectorised `fit / predict_log_proba / predict` in NumPy
- Appreciate why penalising **absent** features distinguishes Bernoulli NB from Multinomial NB

## Problem Statement

Given a labelled training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ with **binary** features $x^{(i)} \in \{0,1\}^n$ and class labels $y^{(i)} \in \{0, \ldots, K{-}1\}$, learn a classifier that assigns the most probable class to a new binary input $x$.

---

### Motivating Example — Text Classification

Represent each document as a **vocabulary-sized binary vector**: $x_j = 1$ if word $j$ appears in the document, $x_j = 0$ otherwise.

| Feature | Value | Meaning |
|---|---|---|
| `x_"money"` | 1 | word *money* is present |
| `x_"free"` | 1 | word *free* is present |
| `x_"meeting"` | 0 | word *meeting* is absent |

Goal: classify as **spam** ($y=1$) or **ham** ($y=0$).

---

### Comparison of Naive Bayes Variants

| Variant | Feature type | Likelihood | Key parameter |
|---|---|---|---|
| **Gaussian NB** | Continuous $\mathbb{R}$ | $\mathcal{N}(\mu_{kj}, \sigma^2_{kj})$ | mean, variance |
| **Multinomial NB** | Integer counts $\mathbb{Z}_{\geq 0}$ | Multinomial | word probabilities |
| **Bernoulli NB** ← *this notebook* | Binary $\{0,1\}$ | Bernoulli | presence probability $\phi_{kj}$ |

## The Multivariate Bernoulli Assumption

Each feature $x_j \in \{0,1\}$ is modelled as an independent **Bernoulli** random variable conditioned on the class:

$$P(x_j \mid y=k) = \phi_{kj}^{x_j}\,(1 - \phi_{kj})^{1 - x_j}$$

where $\phi_{kj} = P(x_j = 1 \mid y=k)$ is the probability that feature $j$ is present in class $k$.

Applying the naïve independence assumption across all $n$ features:

$$P(x \mid y=k) = \prod_{j=1}^{n} \phi_{kj}^{x_j}\,(1 - \phi_{kj})^{1 - x_j}$$

> **Key distinction from Multinomial NB:** Every feature position is explicitly modelled — absent features ($x_j = 0$) contribute the factor $(1 - \phi_{kj})$, actively penalising classes where word $j$ is expected to appear.

## Model / Hypothesis

The MAP decision rule in **log-space**:

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} \log P(x_j \mid y=k) \right]$$

Substituting the Bernoulli likelihood:

$$\hat{y} = \arg\max_{k} \left[ \log \phi_k + \sum_{j=1}^{n} \left( x_j \log \phi_{kj} + (1-x_j) \log(1 - \phi_{kj}) \right) \right]$$

This can be rewritten as a dot product, enabling efficient vectorised computation:

$$\text{score}_k = \log \phi_k + x^\top \log \boldsymbol{\phi}_k + (1-x)^\top \log(1 - \boldsymbol{\phi}_k)$$

where $\boldsymbol{\phi}_k = [\phi_{k1}, \ldots, \phi_{kn}]^\top$.

## Derivation

**Step 1 — Bayes' theorem.**

$$P(y=k \mid x) = \frac{P(x \mid y=k)\, P(y=k)}{P(x)}$$

**Step 2 — Drop the evidence.** $P(x)$ is constant across classes:

$$\hat{y} = \arg\max_{k}\; P(x \mid y=k)\, P(y=k)$$

**Step 3 — Apply Bernoulli factorisation.**

$$\hat{y} = \arg\max_{k}\; P(y=k) \prod_{j=1}^{n} \phi_{kj}^{x_j}(1-\phi_{kj})^{1-x_j}$$

**Step 4 — Take log.**

$$\hat{y} = \arg\max_{k} \left[ \log P(y=k) + \sum_{j=1}^{n} \left( x_j \log \phi_{kj} + (1-x_j)\log(1-\phi_{kj}) \right) \right]$$

**Step 5 — MLE parameter estimates.**

$$\phi_k = \frac{m_k}{m}, \qquad \phi_{kj} = \frac{\sum_{i:\,y^{(i)}=k} x^{(i)}_j}{m_k}$$

**Step 6 — Laplace smoothing** (avoids $\log 0$ when a feature never appears in a class):

$$\phi_{kj} = \frac{\left(\sum_{i:\,y^{(i)}=k} x^{(i)}_j\right) + \alpha}{m_k + 2\alpha}$$

The denominator uses $2\alpha$ because each Bernoulli feature has two outcomes (0 and 1).

## Algorithm

**Training — `fit(X, y, alpha=1)`**

1. Find unique classes $\{0, \ldots, K{-}1\}$
2. For each class $k$, collect $X_k = X[y == k]$, count $m_k = |X_k|$
3. Compute log prior: $\log\phi_k = \log(m_k / m)$
4. Compute smoothed feature probabilities: $\phi_{kj} = (\text{sum}(X_k,\;\text{axis}=0) + \alpha)\;/\;(m_k + 2\alpha)$ — shape $(n,)$
5. Store $\log \boldsymbol{\phi}_k$ and $\log(1 - \boldsymbol{\phi}_k)$ — shape $(K, n)$ each

**Prediction — `predict(X, params)`**

1. $\text{score}_k = \log\phi_k + X \cdot \log\boldsymbol{\phi}_k^\top + (1-X) \cdot \log(1-\boldsymbol{\phi}_k)^\top$ — vectorised $(m, K)$
2. Return $\hat{y} = \arg\max_k\; \text{score}_k$

---

### Shape Reference

| Symbol | Shape | Description |
|---|---|---|
| $X$ | $(m, n)$ | Binary feature matrix |
| $y$ | $(m,)$ | Integer class labels |
| `log_prior` | $(K,)$ | Log class priors |
| `log_phi` | $(K, n)$ | $\log P(x_j=1 \mid y=k)$ |
| `log_1m_phi` | $(K, n)$ | $\log P(x_j=0 \mid y=k)$ |
| `log_scores` | $(m, K)$ | Log posterior score per sample per class |

In [ ]:
import numpy as np


def fit(X, y, alpha=1.0):
    """
    Fit Bernoulli Naive Bayes (Multivariate Bernoulli Event Model).

    Inputs
    ------
    X     : ndarray, (m, n)  binary feature matrix — values in {0, 1}
    y     : ndarray, (m,)    integer class labels 0 .. K-1
    alpha : float            Laplace smoothing pseudo-count (default 1)

    Output
    ------
    params : dict
        log_prior   : ndarray (K,)    log P(y=k)
        log_phi     : ndarray (K, n)  log P(x_j=1 | y=k)  — smoothed
        log_1m_phi  : ndarray (K, n)  log P(x_j=0 | y=k)  — smoothed
    """
    classes = np.unique(y)
    K = len(classes)
    m, n = X.shape

    log_prior  = np.zeros(K)
    log_phi    = np.zeros((K, n))
    log_1m_phi = np.zeros((K, n))

    for k in classes:
        Xk = X[y == k]                         # (m_k, n)
        mk = len(Xk)
        log_prior[k] = np.log(mk / m)
        # Laplace-smoothed feature probabilities
        # denominator is mk + 2*alpha (two outcomes per Bernoulli feature)
        phi_k          = (Xk.sum(axis=0) + alpha) / (mk + 2 * alpha)  # (n,)
        log_phi[k]     = np.log(phi_k)
        log_1m_phi[k]  = np.log(1 - phi_k)

    return dict(log_prior=log_prior, log_phi=log_phi, log_1m_phi=log_1m_phi)


def predict_log_proba(X, params):
    """
    Compute log posterior score for each class.

    Inputs
    ------
    X      : ndarray, (m, n)  binary test features
    params : dict              output of fit()

    Output
    ------
    log_scores : ndarray, (m, K)  unnormalised log posterior per class
    """
    lp        = params['log_prior']   # (K,)
    log_p     = params['log_phi']     # (K, n)
    log_1m_p  = params['log_1m_phi']  # (K, n)
    # score_k = log_prior_k
    #         + X        @ log_phi[k]      (present features)
    #         + (1 - X)  @ log_1m_phi[k]   (absent features)
    # Vectorised over all (m, K) simultaneously:
    return lp + X @ log_p.T + (1 - X) @ log_1m_p.T   # (m, K)


def predict(X, params):
    """
    Predict class labels for test samples.

    Inputs
    ------
    X      : ndarray, (m, n)  binary test features
    params : dict              output of fit()

    Output
    ------
    y_hat : ndarray, (m,)  predicted class labels
    """
    return np.argmax(predict_log_proba(X, params), axis=1)

## Key Properties

| Property | Detail |
|---|---|
| **Type** | Generative classifier — models $P(x, y)$ |
| **Feature type** | Binary $\{0,1\}$ only |
| **Training** | $O(m \cdot n)$ — one pass per class |
| **Prediction** | $O(K \cdot n)$ per sample (matrix multiply) |
| **Absent features** | Explicitly penalised via $(1-\phi_{kj})$ — key difference from Multinomial NB |
| **Laplace smoothing** | $\alpha=1$, denominator $m_k + 2\alpha$ (two Bernoulli outcomes) |
| **Underflow guard** | All likelihoods computed in log-space |
| **Typical use** | Text classification with binary bag-of-words |